# Try notebookproof in five minutes
Run this in your normal notebook environment. Install notebookproof once, then run the cells below.

The base package needs no new runtime dependency. This example uses pandas, which should already be installed in a pandas notebook.

Notebook preflight inspects saved code; it does not execute it or prove reproducibility. DataFrame health inspects the real values without changing the table.

In [ ]:
import notebookproof
import pandas as pd

print("notebookproof", notebookproof.__version__)
df = pd.DataFrame({
    "account_id": [1, 1, 2, 3],
    "amount": pd.Series([10, "10", None, 40], dtype=object),
    "status": ["NA", "NA", "?", None],
})
report = notebookproof.frame(df, keys=["account_id"])
assert report["rows"] == 4
assert report["columns"][2]["missing"] == 1
assert report["duplicateRows"] == 0

## Inspect deliberately broken saved code
The next cell creates a temporary sample notebook. Its saved code uses a value before the definition and contains an old error output. The checker reads that sample without executing it. Cell numbers include Markdown.

On your own notebook, save first and call `notebookproof.check("your-notebook.ipynb")`. Relative paths use your kernel's working directory.

In [ ]:
import json
import tempfile
from pathlib import Path

sample = {"cells": [
    {"cell_type": "markdown", "source": "# Example"},
    {"cell_type": "code", "source": "print(total)", "execution_count": 3,
     "outputs": [{"output_type": "error", "ename": "NameError", "evalue": "total is not defined"}]},
    {"cell_type": "code", "source": "total = 42", "execution_count": 1, "outputs": []},
]}
with tempfile.TemporaryDirectory() as folder:
    sample_path = Path(folder) / "sample.ipynb"
    sample_path.write_text(json.dumps(sample), encoding="utf-8")
    preflight = notebookproof.check(sample_path)

assert preflight["executed"] is False
assert {"FORWARD_REF", "OUT_OF_ORDER", "SAVED_ERROR"} <= {f["code"] for f in preflight["findings"]}

## Optional: run a fresh kernel
Install the optional execution extra, `notebookproof[verify]`, then call:

```python
result = notebookproof.verify("your-notebook.ipynb", timeout=60)
```

This executes your saved code and its side effects with your usual permissions in another kernel. It leaves the original notebook file untouched and does not restart your current kernel. It reports execution success or failure; it does not compare outputs or guarantee reproducibility.

Only execute notebooks you intend to run. Nested `verify()` calls are skipped to prevent recursive kernels.

## Tell us what was useful
Did this catch something you had missed? Were any findings wrong or confusing? Would you use it again before sharing a notebook or reporting a number? Recruiting for the five-person trial will happen later.